In [37]:
%cd /workspace/EBES/

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from pathlib import Path
import optuna 
from ebes.pipeline.utils import optuna_df
from optuna.trial import TrialState

/workspace/EBES


/usr/local/lib/python3.10/dist-packages/IPython/core/magics/osm.py:417: UserWarning:

using dhist requires you to install the `pickleshare` library.



In [38]:
#%pip install plotly

In [42]:
def get_run(number, specify="best", rewrite=False):
    path = Path(f"log/{dataset}/{method}/optuna/{number}")
    print(pd.read_csv(path / "results.csv"))
    print((path / "params.txt").read_text())
    save_path = Path(f"configs/specify/{dataset_ntp}/{method_emb}")
    save_path.mkdir(parents=True, exist_ok=True)
    save_path = (save_path / f"{specify}.yaml")
    if not rewrite:
        assert not save_path.exists()
    save_path.write_text((path / "params.txt").read_text())

def prepare_data(dataset, method):
    path = Path(f"log/{dataset}/{method}/optuna")
    df, study = optuna_df(path)
    col_to_drop = ["datetime_start", "datetime_complete", "system_attrs_fixed_params", "state", "value"]
    col_params = ["value", "duration"] + [col for col in df if "params_" in col]
    col_user = ["value", "duration"] + [col for col in df if "user" in col]
    df["duration"] = df["duration"].dt.total_seconds()
    return df, study, col_user, col_params

In [43]:
dataset = "rossman"
method = "coles_ntp"
df, study, col_user, col_params = prepare_data(dataset, method)

print(df.shape, df[~df["value"].isna()].shape)
test_cols = [col for col in col_user if ("test" in col)]
df[~df["value"].isna()][:50].sort_values("value").iloc[:10, ][["value"] + test_cols]

(59, 24) (56, 24)


/workspace/EBES/ebes/pipeline/utils.py:161: ExperimentalWarning:

JournalStorage is experimental (supported from v3.1.0). The interface can change in the future.



,value,user_attrs_test_loss_mean,user_attrs_test_loss_std
32,19.723804,4547.944165,0.0
24,39.454987,6538.976318,0.0
18,43.083000,5634.816113,0.0
22,44.611652,7115.399731,0.0
15,48.005402,8249.026172,0.0
47,50.291550,993.433633,0.0
42,67.518013,15100.571387,0.0
40,74.713737,14736.852295,0.0
30,120.914780,3513.600842,0.0
23,123.996010,1757.808069,0.0


In [44]:
get_run(32, specify="best", rewrite=True)

     Unnamed: 0            0         mean  std
0    train_loss    25.754726    25.754726  0.0
1          loss    19.723804    19.723804  0.0
2     test_loss  4547.944165  4547.944165  0.0
3  memory_after  2912.000000  2912.000000  0.0
optimizer:
  params:
    weight_decay: 1.4303270081812846e-05
    lr: 0.0001381744310626628
model:
  encoder:
    params:
      num_layers: 3
      dropout: 0.015207866009611692
  preprocess:
    params:
      time_process: diff
      num_norm: true
      cat_emb_dim: 118
      num_emb_dim: 43
  aggregation:
    name: ValidHiddenMean
unsupervised_loss:
  params:
    margin: 0.24195278138817267



In [45]:
failed = df[(df["state"] != "COMPLETE") | (df[col_user].isna().any(axis=1))][col_user].index
print(failed)
for fail in df[(df["state"] != "COMPLETE") | (df[col_user].isna().any(axis=1))][col_user].index:
    error_path = Path(f"/home/dev/24/es-bench/log/{dataset}/{method}/optuna/{fail}/ERROR.txt")
    if error_path.exists():
        error = error_path.read_text()
        print(fail, error.split("\n")[-2])
    else:
        print(df.loc[fail])

Index([3, 4, 58], dtype='int64')
value                                                                 NaN
datetime_start                                 2026-01-23 13:33:10.521341
datetime_complete                                                     NaT
duration                                                              NaN
params_model.aggregation.name                             ValidHiddenMean
params_model.encoder.params.dropout                                   0.0
params_model.encoder.params.num_layers                                  3
params_model.preprocess.params.cat_emb_dim                             93
params_model.preprocess.params.num_emb_dim                              6
params_model.preprocess.params.num_norm                              True
params_model.preprocess.params.time_process                          diff
params_optimizer.params.lr                                       0.001055
params_optimizer.params.weight_decay                                  0.0
param

In [32]:
optuna.visualization.plot_optimization_history(study)

### Params influence

In [33]:
trials = study.trials
trials = [trial for trial in trials if trial.state == TrialState.COMPLETE]
plotted_trials = sorted(trials, key=lambda t: t.value)[:]
plotted_study = optuna.create_study()
for trial in plotted_trials:
    plotted_study.add_trial(trial)

[I 2026-01-29 08:06:03,494] A new study created in memory with name: no-name-3b3b0cd0-9bdb-4ebf-b5a1-80774f55a930


In [34]:
target = None #lambda t: (t.user_attrs["memory_after_mean"])
target_name = "value"
fig = optuna.visualization.plot_param_importances(plotted_study, target=target, target_name=target_name)
print(fig._data[0]["x"][::-1])
print(fig._data[0]["y"][::-1])
take = 7
params = fig._data[0]["y"][-take:]
not_imp = list(set([col.replace("params_", "") for col in col_params]) - set(params) - {"duration", "value", "system_attrs_fixed_params"})
fig

[0.3470781014841039, 0.24160830915341516, 0.132108140417412, 0.10342374536673926, 0.09702384772446161, 0.06382672633609178, 0.009083805230004348, 0.004524537052408822, 0.0012250815039332806, 9.770573142982166e-05]
['model.preprocess.params.cat_emb_dim', 'unsupervised_loss.params.margin', 'model.preprocess.params.num_norm', 'model.encoder.params.dropout', 'model.preprocess.params.num_emb_dim', 'optimizer.params.lr', 'model.encoder.params.num_layers', 'model.preprocess.params.time_process', 'model.aggregation.name', 'optimizer.params.weight_decay']


In [35]:
params

['model.encoder.params.num_layers',
 'optimizer.params.lr',
 'model.preprocess.params.num_emb_dim',
 'model.encoder.params.dropout',
 'model.preprocess.params.num_norm',
 'unsupervised_loss.params.margin',
 'model.preprocess.params.cat_emb_dim']

In [36]:
# fig = optuna.visualization.plot_parallel_coordinate(plotted_study, target=target, target_name=target_name, params=['model.encoder.params.pooling', 'pretrain_model.encoder.params.pooling',])
# fig = optuna.visualization.plot_contour(study, target=target, target_name=target_name, params=params+not_imp)
fig = optuna.visualization.plot_slice(study, target=target, target_name=target_name)#, params=["model.encoder.params.num_layers"] )
# fig = optuna.visualization.plot_optimization_history(study, target=target, target_name=target_name, error_bar=False)
# targets = lambda t: (t.user_attrs["memory_after_mean"], t.user_attrs["val_metric_mean"])
# target_names = ["memory_after_mean", "val_metric_mean"]
# fig = optuna.visualization.plot_pareto_front(study, targets=targets, target_names=target_names)
fig